##Build a Recursive Language Model (RLM) from Scratch

By the end of this notebook, you will have:
1. **Seen long-context failure** in action — a model failing on a simple task with too much context
2. **Built an RLM from scratch** — the REPL, the agent loop, and recursive sub-LLM calls
3. **Compared vanilla LLM vs RLM** on the same task

Based on the paper: [Recursive Language Models](https://arxiv.org/abs/2512.24601) by Alex Zhang, Tim Kraska & Omar Khattab (MIT CSAIL)

---

###Setup

We use OpenRouter for free model access, same as the agents class.

In [3]:
import requests
import json
import re
import io
import sys
import traceback
from contextlib import redirect_stdout, redirect_stderr
from google.colab import userdata

OPENROUTER_API_KEY = userdata.get('OPENROUTER_API_KEY')
# Models — use cheap/free ones for the class
ROOT_MODEL = "google/gemini-3-flash-preview"  # root agent (stronger)
SUB_MODEL  = "google/gemini-3-flash-preview"  # sub-agent (can be cheaper)

def llm_call(prompt, system="", model=ROOT_MODEL, max_tokens=4000):
    """Call an LLM via OpenRouter. Returns the text response."""
    msgs = []
    if system:
        msgs.append({"role": "system", "content": system})
    msgs.append({"role": "user", "content": prompt})

    r = requests.post(
        "https://openrouter.ai/api/v1/chat/completions",
        headers={"Authorization": f"Bearer {OPENROUTER_API_KEY}"},
        json={"model": model, "messages": msgs, "max_tokens": max_tokens}
    )
    data = r.json()
    if "choices" not in data:
        raise Exception(f"API error: {data}")
    return data["choices"][0]["message"]["content"]

# Quick test
print(llm_call("Say hello in one word."))

Hello.


---
## Part 1: The Problem — Long Context Fails

Before building a Recursive Language Model (RLM), let's first understand the challenge. Large Language Models (LLMs) can process long documents, but as the context grows, finding specific information becomes increasingly difficult.

In this section, we'll generate a large synthetic dataset containing thousands of records. Our task is simple: **count how many engineers are based in Tokyo**. While this is easy to solve with code, it becomes a useful benchmark for evaluating how well an LLM handles long-context reasoning.

By the end of this section, you will:
- Generate a large synthetic dataset.
- Create a long-context document.
- Calculate the ground truth for later comparison with the LLM.

In [4]:
import random
random.seed(42)

# ──────────────────────────────────────────────
# Generate a synthetic dataset: people with cities and professions
# ──────────────────────────────────────────────

first_names = ["Alice", "Bob", "Charlie", "Diana", "Eve", "Frank",
               "Grace", "Hank", "Ivy", "Jack", "Karen", "Leo",
               "Mona", "Nate", "Olivia", "Paul", "Quinn", "Rita",
               "Sam", "Tina", "Uma", "Vince", "Wendy", "Xander", "Yara", "Zane"]

cities = ["New York", "London", "Tokyo", "Paris", "Berlin",
          "Mumbai", "Sydney", "Toronto", "Dubai", "Singapore",
          "Seoul", "Bangkok", "Cairo", "Lagos", "Lima"]

professions = ["engineer", "doctor", "teacher", "artist", "chef",
               "pilot", "lawyer", "nurse", "writer", "musician"]

def generate_dataset(n_entries=500):
    """Generate a dataset of people with names, cities, and professions."""
    entries = []
    for i in range(n_entries):
        name = f"{random.choice(first_names)} {random.choice('ABCDEFGHIJKLMNOPQRSTUVWXYZ')}."
        city = random.choice(cities)
        prof = random.choice(professions)
        age = random.randint(22, 65)
        entries.append(f"Entry {i+1}: {name}, age {age}, {prof}, based in {city}")
    return entries

entries = generate_dataset(5000)
dataset_text = "\n".join(entries)

# Ground truth: count engineers in Tokyo
ground_truth = sum(1 for e in entries if "engineer" in e and "Tokyo" in e)

print(f"Dataset: {len(entries)} entries, {len(dataset_text)} characters")
print(f"\nFirst 5 entries:")
for e in entries[:5]:
    print(f"  {e}")
print(f"\n🎯 Ground truth: {ground_truth} engineers in Tokyo")

Dataset: 5000 entries, 265745 characters

First 5 entries:
  Entry 1: Uma D., age 37, chef, based in New York
  Entry 2: Hank E., age 65, doctor, based in Bangkok
  Entry 3: Xander R., age 49, musician, based in London
  Entry 4: Bob A., age 36, artist, based in London
  Entry 5: Quinn T., age 34, writer, based in New York

🎯 Ground truth: 40 engineers in Tokyo


### Try it with a vanilla LLM — just stuff the context in

In [5]:
# ──────────────────────────────────────────────
# VANILLA APPROACH: stuff everything into the prompt
# ──────────────────────────────────────────────

vanilla_prompt = f"""Here is a dataset of people. Count EXACTLY how many are engineers in Tokyo.
Return ONLY the number, nothing else.

{dataset_text}"""

print(f"Prompt length: {len(vanilla_prompt)} characters")
print("Asking vanilla LLM...")

vanilla_answer = llm_call(vanilla_prompt)
print(f"\nVanilla LLM answer: {vanilla_answer}")
print(f"Ground truth:        {ground_truth}")
print(f"Correct?             {'✅ Yes' if str(ground_truth) in vanilla_answer else '❌ No'}")

Prompt length: 265860 characters
Asking vanilla LLM...

Vanilla LLM answer: 21
Ground truth:        40
Correct?             ❌ No


### It given wrong


Even if the model gets this one right with 500 entries, the approach doesn't scale.
At 5,000 or 50,000 entries, context rot destroys accuracy.

**The RLM approach: instead of feeding all 500 entries to the model, let the model write code to count them itself.**

---
## Part 2: Build the REPL Environment

The REPL is where the model's code runs. We need:
- The context stored as a variable (not in the prompt)
- `print()` output captured and returned to the model
- A `FINAL()` function to signal the answer
- A `llm_query()` function for recursive sub-LLM calls

In [6]:
import io
import re
import json

from contextlib import redirect_stdout, redirect_stderr

# ──────────────────────────────────────────────
# THE REPL: A Python Execution Environment
# ──────────────────────────────────────────────

class RLMRepl:
    """A REPL environment for Recursive Language Models (RLM).

    The long context is stored as a Python variable instead of being
    placed inside the model's prompt. The model writes Python code,
    executes it, observes the output, and continues reasoning.
    """

    def __init__(self, context: str, max_output_chars: int = 5000):
        self.final_answer = None
        self.max_output_chars = max_output_chars
        self.sub_call_count = 0

        # Namespace available while executing model-generated code.
        self.namespace = {
            "context": context,
            "FINAL": self._final,
            "llm_query": self._llm_query,

            # Useful Python modules
            "re": re,
            "json": json,

            # Common built-in functions
            "len": len,
            "print": print,
            "int": int,
            "float": float,
            "str": str,
            "list": list,
            "dict": dict,
            "range": range,
            "enumerate": enumerate,
            "sum": sum,
            "sorted": sorted,
            "min": min,
            "max": max,
            "abs": abs,
            "set": set,
            "tuple": tuple,
            "zip": zip,
            "map": map,
            "filter": filter,
            "isinstance": isinstance,
            "type": type,
            "True": True,
            "False": False,
            "None": None,
        }

    def _final(self, answer):
        """Store the model's final answer."""
        self.final_answer = str(answer)
        print(f"[FINAL ANSWER SUBMITTED: {answer}]")

    def _llm_query(self, query: str, sub_context: str = "") -> str:
        """Call a sub-LLM for recursive reasoning."""

        self.sub_call_count += 1

        print(f"  [Sub-LLM Call #{self.sub_call_count}]")

        prompt = query
        if sub_context:
            prompt = f"{query}\n\nContext:\n{sub_context}"

        result = llm_call(
            prompt,
            model=SUB_MODEL,
            max_tokens=2000
        )

        print("  Sub-LLM completed successfully.")

        return result

    def execute(self, code: str) -> str:
        """Execute Python code and capture stdout."""

        stdout_capture = io.StringIO()
        stderr_capture = io.StringIO()

        try:
            with redirect_stdout(stdout_capture), redirect_stderr(stderr_capture):
                exec(code, self.namespace)

            output = stdout_capture.getvalue()

        except Exception as e:
            output = f"ERROR: {type(e).__name__}: {e}"

        # Prevent extremely large outputs
        if len(output) > self.max_output_chars:
            output = (
                output[:self.max_output_chars]
                + f"\n... [Output truncated to {self.max_output_chars} characters]"
            )

        return output


# ──────────────────────────────────────────────
# Quick Test of the REPL
# ──────────────────────────────────────────────

repl = RLMRepl(
    context="Hello world! This is a test context with some data."
)

print("Test 1 — Access Context")
print(repl.execute("print(context[:30])"))

print("\nTest 2 — Count Words")
print(
    repl.execute(
        """
import re
words = re.findall(r"\\w+", context)
print(f"Words: {len(words)}")
"""
    )
)

print("\nTest 3 — Submit Final Answer")
print(repl.execute("FINAL(42)"))

print(f"Stored Answer: {repl.final_answer}")

print("\n✅ REPL is working correctly!")

Test 1 — Access Context
Hello world! This is a test co


Test 2 — Count Words
Words: 10


Test 3 — Submit Final Answer
[FINAL ANSWER SUBMITTED: 42]

Stored Answer: 42

✅ REPL is working correctly!


### Explanation

The `RLMRepl` class creates a custom **Python REPL (Read-Eval-Print Loop)** where the Recursive Language Model (RLM) can execute Python code instead of relying only on text generation. This allows the model to reason step by step, inspect intermediate results, and solve tasks more effectively.

During initialization (`__init__`), the long input is stored in the `context` variable inside the REPL's namespace. This is a key idea behind RLMs—the large context is kept in memory rather than being included in the LLM's prompt. The namespace also exposes useful functions like `FINAL()` for submitting the final answer, `llm_query()` for making recursive sub-LLM calls, and common Python modules and built-in functions that the model may need.

The `execute()` method is responsible for running the Python code generated by the model. It captures everything printed using `print()` and returns it as output, allowing the model to observe the result of its execution. If an error occurs, the exception is caught and returned instead of stopping the program. To prevent overwhelming the model, very large outputs are automatically truncated.

Finally, a few simple tests verify that the REPL works correctly. We check that the model can access the stored context, execute Python code, and submit a final answer using `FINAL()`. These tests ensure the REPL is ready for building and experimenting with Recursive Language Models.

---
## Part 3: Build the RLM Agent Loop

With the REPL environment ready, we can now build the **RLM Agent Loop**, which is the core component of a Recursive Language Model. Instead of generating a single answer, the model repeatedly writes Python code, executes it inside the REPL, observes the output, and uses that feedback to decide its next action.

This iterative **reason → execute → observe → refine** process continues until the model calls `FINAL()` with the correct answer or reaches the maximum number of iterations. Unlike a traditional LLM, the long context remains inside the REPL and is never sent directly in the prompt, making this approach more scalable for long-context reasoning.

In [9]:
# ──────────────────────────────────────────────
# THE RLM SYSTEM PROMPT
# This tells the model HOW to use the REPL
# ──────────────────────────────────────────────

RLM_SYSTEM_PROMPT = """You are an RLM (Recursive Language Model) agent.

You have access to a Python REPL environment. The user's data is stored
in a variable called `context` — it may be very long (millions of characters).
You CANNOT see the context directly. You must write Python code to explore it.

Available tools:
- `context` — the full input text (Python string variable)
- `print()` — use this to see output from your code
- `llm_query(query, sub_context)` — call a sub-LLM to analyze a chunk.
  The sub-LLM's response is returned as a string. It does NOT enter your context.
- `FINAL(answer)` — call this when you have the final answer.
- Standard Python: `re`, `json`, `len`, `sum`, etc.

Strategy:
1. First, check the size: `print(len(context))`
2. Peek at the structure: `print(context[:500])`
3. Use code to search, filter, count, or slice the data
4. For complex subtasks, use `llm_query()` to delegate to a sub-LLM
5. When done, call `FINAL(your_answer)`

Rules:
- Write ONLY Python code. No markdown, no explanation.
- Your code block must be wrapped in ```python ... ```
- Use print() to see results — you only see what you print.
- Variables persist between steps (like Jupyter cells).
- Be systematic. Explore first, then solve.
"""

print("System prompt defined ✅")

System prompt defined ✅


In [10]:
# ──────────────────────────────────────────────
# THE RLM LOOP
# ──────────────────────────────────────────────

def extract_code(response: str) -> str:
    """Extract Python code from the LLM's response."""
    # Try to find ```python ... ``` blocks
    pattern = r'```python\s*\n(.*?)```'
    matches = re.findall(pattern, response, re.DOTALL)
    if matches:
        return matches[0].strip()

    # Try ``` ... ``` blocks
    pattern = r'```\s*\n(.*?)```'
    matches = re.findall(pattern, response, re.DOTALL)
    if matches:
        return matches[0].strip()

    # If no code blocks, try to use the whole response as code
    # (strip any leading text before first line that looks like code)
    lines = response.strip().split('\n')
    code_lines = []
    started = False
    for line in lines:
        if not started and line.startswith(('import ', 'from ', 'print(', '#', 'for ', 'if ', 'def ', 'context', 'result', 'count', 'data', 'lines', 'FINAL')):
            started = True
        if started:
            code_lines.append(line)

    return '\n'.join(code_lines) if code_lines else response.strip()


def run_rlm(query: str, context: str, max_iterations: int = 10, verbose: bool = True):
    """Run an RLM agent on a query with a given context.

    The context is NOT sent to the LLM. It's stored in the REPL
    as a Python variable. The LLM writes code to explore it.

    Args:
        query: The question to answer
        context: The (potentially huge) input text
        max_iterations: Max REPL interaction loops
        verbose: Print each step

    Returns:
        dict with 'answer', 'iterations', 'sub_calls', 'history'
    """
    repl = RLMRepl(context)
    history = []  # conversation history for the root LLM

    # First message: just the query (NOT the context!)
    user_msg = f"""Task: {query}

The data is in the `context` variable ({len(context)} characters long).
Write Python code to explore and solve this. Start by checking the structure."""

    history.append({"role": "user", "content": user_msg})

    for i in range(max_iterations):
        if verbose:
            print(f"\n{'='*60}")
            print(f"  ITERATION {i+1}/{max_iterations}")
            print(f"{'='*60}")

        # Ask the LLM to write code
        response = requests.post(
            "https://openrouter.ai/api/v1/chat/completions",
            headers={"Authorization": f"Bearer {OPENROUTER_API_KEY}"},
            json={
                "model": ROOT_MODEL,
                "messages": [{"role": "system", "content": RLM_SYSTEM_PROMPT}] + history,
                "max_tokens": 2000
            }
        ).json()

        assistant_msg = response["choices"][0]["message"]["content"]
        history.append({"role": "assistant", "content": assistant_msg})

        # Extract code from the response
        code = extract_code(assistant_msg)

        if verbose:
            print(f"\n📝 LLM wrote:\n")
            for line in code.split('\n'):
                print(f"    {line}")

        # Execute in the REPL
        output = repl.execute(code)

        if verbose:
            print(f"\n📤 REPL output:\n")
            for line in output.split('\n'):
                print(f"    {line}")

        # Check if FINAL was called
        if repl.final_answer is not None:
            if verbose:
                print(f"\n🏁 FINAL ANSWER: {repl.final_answer}")
                print(f"   Iterations: {i+1}")
                print(f"   Sub-LLM calls: {repl.sub_call_count}")
            return {
                "answer": repl.final_answer,
                "iterations": i + 1,
                "sub_calls": repl.sub_call_count,
                "history": history
            }

        # Feed the output back to the LLM for the next iteration
        history.append({
            "role": "user",
            "content": f"REPL output:\n```\n{output}\n```\nContinue. Write more code or call FINAL(answer) when done."
        })

    if verbose:
        print(f"\n⚠️ Max iterations reached without FINAL()")
    return {
        "answer": None,
        "iterations": max_iterations,
        "sub_calls": repl.sub_call_count,
        "history": history
    }

print("RLM agent loop defined ✅")

RLM agent loop defined ✅


### Explanation

The `extract_code()` function extracts executable Python code from the LLM's response. It first looks for code enclosed in Markdown code blocks (```python```), and if none are found, it attempts to identify code directly from the response. This ensures that only valid Python code is sent to the REPL.

The `run_rlm()` function implements the complete Recursive Language Model loop. It creates a new `RLMRepl` instance, sends the user's query to the LLM, extracts the generated code, executes it in the REPL, and captures the output. The execution results are then fed back to the LLM, allowing it to iteratively improve its reasoning. This process repeats until the model calls `FINAL()` with the answer or the maximum number of iterations is reached.

Unlike a traditional LLM, the long context is never included in the prompt. Instead, it is stored inside the REPL as the `context` variable, and the model interacts with it by writing Python code. This separation of **reasoning** (handled by the LLM) and **data processing** (handled by the REPL) is the key idea behind Recursive Language Models.

---
## Part 4: Testing the RLM

Now, let's see if the RLM approach solves our long-context problem. We'll ask the RLM to perform the same task: count the number of engineers in Tokyo.

In [11]:
# ──────────────────────────────────────────────
# RUNNING THE RLM
# ──────────────────────────────────────────────

query = "Count exactly how many people are engineers based in Tokyo."

print(f"Target: {ground_truth} engineers in Tokyo\n")

rlm_results = run_rlm(query, dataset_text, max_iterations=5)

print("\n" + "="*60)
print("RESULTS COMPARISON")
print("="*60)
print(f"Ground Truth:      {ground_truth}")
print(f"Vanilla LLM:       {vanilla_answer}")
print(f"RLM Agent:         {rlm_results['answer']}")
print(f"\nRLM correctly?     {'✅ Yes' if str(ground_truth) == str(rlm_results['answer']) else '❌ No'}")

Target: 40 engineers in Tokyo


  ITERATION 1/5

📝 LLM wrote:

    print(f"Context length: {len(context)}")
    print("First 500 characters:")
    print(context[:500])

📤 REPL output:

    Context length: 265745
    First 500 characters:
    Entry 1: Uma D., age 37, chef, based in New York
    Entry 2: Hank E., age 65, doctor, based in Bangkok
    Entry 3: Xander R., age 49, musician, based in London
    Entry 4: Bob A., age 36, artist, based in London
    Entry 5: Quinn T., age 34, writer, based in New York
    Entry 6: Wendy U., age 48, writer, based in Bangkok
    Entry 7: Hank O., age 22, chef, based in Singapore
    Entry 8: Yara Z., age 43, lawyer, based in Tokyo
    Entry 9: Ivy E., age 28, pilot, based in Paris
    Entry 10: Charlie M., age 44, pilot, based in
    

  ITERATION 2/5

📝 LLM wrote:

    import re
    
    # The structure appears to be "Entry N: Name, age X, profession, based in City"
    # I will use a regular expression to find all instances of 'engineer' and 'To

### 🔍 Analysis: Why did the RLM win?

As seen in the results above, the **Vanilla LLM** significantly undercounted the engineers (getting 21 instead of 40), while the **RLM** was 100% accurate. Here is why:

1. **Attention Decay (Lost in the Middle):** The Vanilla LLM had to process ~265,000 characters at once. LLMs often lose track of specific details in very large prompts, leading to high error rates in counting tasks.
2. **Reasoning vs. Computation:** The RLM didn't try to "read" and "count" in its head. It performed **Reasoning** to understand the data structure and then generated a **Python script** to handle the **Computation**.
3. **Scalability:** The RLM only sent a few hundred characters to the LLM (the code and logic), keeping the massive dataset localized in the REPL. This makes it far more robust as the dataset grows to millions of lines.

---
##  How RLM Solved the Vanilla Problem

To summarize what we've learned, here is the technical breakdown of how the RLM architecture overcame the failure of the standard LLM call.

### 1. Data Decoupling
In the **Vanilla** approach, the data is part of the "Thinking Space" (the prompt). When the thinking space is cluttered with 260,000 characters of raw data, the model's ability to follow instructions degrades.
In the **RLM**, the data is moved to a **"Storage Space"** (the `context` variable in the REPL). The model's prompt remains clean, containing only instructions and logic.

### 2. Systematic Exploration vs. One-Shot Guessing
Instead of trying to calculate the answer in one pass, the RLM followed a systematic loop:
* **Step 1: Sampling.** It used `print(context[:500])` to identify the pattern of the data.
* **Step 2: Logic Mapping.** It recognized the schema: `Entry N: [Name], age [Age], [Profession], based in [City]`.
* **Step 3: Programmatic Filtering.** It wrote a Python regex to filter for `engineer` and `Tokyo` simultaneously.

### 3. Computation Offloading
LLMs are notorious for being poor at counting large sets. By writing Python code, the model converted a **probabilistic task** (guessing a count) into a **deterministic task** (running a script).

**The Result:**
* **Vanilla:** High cost (265k tokens), Low accuracy (~52%).
* **RLM:** Low cost (~2k tokens), 100% accuracy (40/40).

---
## Part 5: Recursive Reasoning with Sub-LLMs

The power of an RLM isn't just running code—it's the ability to call other LLMs recursively.

In this example, we will use Python to find a specific person in the dataset, and then use `llm_query` to ask a sub-LLM to 'summarize their profile' or 'predict their interests' based on the raw text. This simulates a workflow where code handles the **retrieval** and the sub-LLM handles the **reasoning** over that specific chunk.

In [12]:
# ──────────────────────────────────────────────
# RECURSIVE CALL DEMO
# ──────────────────────────────────────────────

recursive_query = """
1. Find the first person in the context based in 'Paris' who is a 'pilot'.
2. Use llm_query to write a 1-sentence creative bio for them.
3. Return the result with FINAL().
"""

print("Starting Recursive RLM Task...\n")
recursive_results = run_rlm(recursive_query, dataset_text, max_iterations=5)

print(f"\nFinal Result from Sub-LLM delegation: {recursive_results['answer']}")

Starting Recursive RLM Task...


  ITERATION 1/5

📝 LLM wrote:

    print(len(context))
    print(context[:1000])

📤 REPL output:

    265745
    Entry 1: Uma D., age 37, chef, based in New York
    Entry 2: Hank E., age 65, doctor, based in Bangkok
    Entry 3: Xander R., age 49, musician, based in London
    Entry 4: Bob A., age 36, artist, based in London
    Entry 5: Quinn T., age 34, writer, based in New York
    Entry 6: Wendy U., age 48, writer, based in Bangkok
    Entry 7: Hank O., age 22, chef, based in Singapore
    Entry 8: Yara Z., age 43, lawyer, based in Tokyo
    Entry 9: Ivy E., age 28, pilot, based in Paris
    Entry 10: Charlie M., age 44, pilot, based in London
    Entry 11: Tina I., age 51, engineer, based in Cairo
    Entry 12: Rita D., age 27, lawyer, based in Lima
    Entry 13: Rita J., age 45, musician, based in Lagos
    Entry 14: Sam G., age 24, doctor, based in Bangkok
    Entry 15: Vince H., age 27, chef, based in Cairo
    Entry 16: Hank D., age 51, chef, 

---
## Part 6: The Stress Test — Harder Dataset

To really see the RLM shine, we'll create a 'messy' dataset.
- Some entries are plain strings.
- Some entries are JSON blobs.
- Information is scattered (e.g., 'Location' vs 'based in').
- It includes 'red herrings' (e.g., people who *used* to live in Tokyo but moved).

In [13]:
import json
import random

def generate_hard_dataset(n=1000):
    hard_entries = []
    for i in range(n):
        name = f"{random.choice(first_names)} {random.choice('ABCDEFGHIJKLMNOPQRSTUVWXYZ')}."
        city = random.choice(cities)
        prof = random.choice(professions)

        # Create messy formats
        format_type = random.random()
        if format_type < 0.3:
            # Standard format
            hard_entries.append(f"ID_{i}: {name} is a {prof} in {city}.")
        elif format_type < 0.6:
            # JSON format
            data = {"uid": i, "name": name, "job": prof, "loc": city}
            hard_entries.append(json.dumps(data))
        else:
            # Narrative/Messy format
            hard_entries.append(f"Log {i} >> Subject: {name}; Profession: {prof}; Current Residence: {city}. Previously: Berlin.")

    return "\n".join(hard_entries)

hard_dataset_text = generate_hard_dataset(5000)
print(f"Hard Dataset Generated: {len(hard_dataset_text)} characters.")
print("\nSample of messy data:")
print("\n".join(hard_dataset_text.split('\n')[:5]))

Hard Dataset Generated: 354245 characters.

Sample of messy data:
{"uid": 0, "name": "Uma L.", "job": "nurse", "loc": "Berlin"}
ID_1: Eve G. is a doctor in Lagos.
ID_2: Grace E. is a nurse in Lima.
Log 3 >> Subject: Rita T.; Profession: nurse; Current Residence: Cairo. Previously: Berlin.
{"uid": 4, "name": "Tina X.", "job": "doctor", "loc": "Lagos"}


### Part 7: Running RLM on the Hard Dataset

Now we challenge the RLM to count engineers in Tokyo within the `hard_dataset_text`.

**The Challenge:** The model must now handle multiple formats simultaneously:
1.  `{"job": "engineer", "loc": "Tokyo"}`
2.  `ID_N: Name is a engineer in Tokyo.`
3.  `Log N >> Subject: Name; Profession: engineer; Current Residence: Tokyo.`

In [14]:
# Calculate ground truth for the hard dataset first
def calculate_hard_ground_truth(text):
    count = 0
    for line in text.split('\n'):
        l = line.lower()
        if 'engineer' in l and 'tokyo' in l:
            # Basic validation to ensure it's not a 'previously lived in' red herring
            if 'previously: tokyo' not in l:
                count += 1
    return count

hard_ground_truth = calculate_hard_ground_truth(hard_dataset_text)

query_hard = "Count exactly how many people are engineers currently based in Tokyo. Note: the data has multiple formats (JSON, ID lines, and Logs)."

print(f"Target (Ground Truth): {hard_ground_truth}\n")

hard_rlm_results = run_rlm(query_hard, hard_dataset_text, max_iterations=6)

print("\n" + "="*60)
print("FINAL STRESS TEST RESULTS")
print("="*60)
print(f"Ground Truth: {hard_ground_truth}")
print(f"RLM Agent:    {hard_rlm_results['answer']}")
print(f"Correct?      {'✅ Yes' if str(hard_ground_truth) == str(hard_rlm_results['answer']) else '❌ No'}")

Target (Ground Truth): 42


  ITERATION 1/6

📝 LLM wrote:

    print(f"Total length: {len(context)}")
    print("Sample start:\n", context[:1000])
    print("\nSample end:\n", context[-1000:])

📤 REPL output:

    Total length: 354245
    Sample start:
     {"uid": 0, "name": "Uma L.", "job": "nurse", "loc": "Berlin"}
    ID_1: Eve G. is a doctor in Lagos.
    ID_2: Grace E. is a nurse in Lima.
    Log 3 >> Subject: Rita T.; Profession: nurse; Current Residence: Cairo. Previously: Berlin.
    {"uid": 4, "name": "Tina X.", "job": "doctor", "loc": "Lagos"}
    ID_5: Yara M. is a artist in Cairo.
    {"uid": 6, "name": "Jack U.", "job": "teacher", "loc": "Seoul"}
    {"uid": 7, "name": "Alice G.", "job": "writer", "loc": "Lagos"}
    Log 8 >> Subject: Vince F.; Profession: doctor; Current Residence: Lagos. Previously: Berlin.
    ID_9: Leo W. is a writer in New York.
    Log 10 >> Subject: Charlie T.; Profession: lawyer; Current Residence: Cairo. Previously: Berlin.
    ID_11: Bob I. is a

## Conclusion: The RLM Advantage

In this notebook, we've demonstrated that Recursive Language Models solve the biggest bottleneck of modern LLMs: **The Reliability-Context Tradeoff.**

### Key Takeaways:
1. **Deterministic Accuracy:** By writing Python code, the model avoids the fuzzy "guessing" of token prediction for mathematical or counting tasks.
2. **Format Agnostic:** The RLM successfully navigated a mix of JSON, ID strings, and narrative logs by iteratively refining its parsing logic based on REPL feedback.
3. **Efficient Reasoning:** The model used less than 1% of the potential token cost compared to a 'vanilla' long-context call, while achieving 100% accuracy vs. the vanilla model's ~50% failure rate.

### Next Steps:
You can extend this RLM by adding more tools to the `RLMRepl` namespace, such as `pandas` for tabular data or `BeautifulSoup` for HTML parsing, allowing the agent to tackle even more complex real-world data environments.

## 🏁 Final Recap & Paper Insights

### 1. Notebook Summary
In this session, we successfully:
- **Identified Long-Context Failure:** Observed a standard LLM fail with 52% error rate on a simple counting task.
- **Built an RLM from Scratch:** Created a custom REPL and an agent loop that offloads data processing to Python.
- **Achieved 100% Accuracy:** Used the RLM to solve both standard and messy (JSON/Log) datasets flawlessly.
- **Demonstrated Recursion:** Used sub-LLM calls for qualitative tasks while keeping the root agent focused on logic.

### 2. Core Results from the RLM Paper
The paper *Recursive Language Models* (Zhang et al., 2025) establishes that:
- **Infinite Context:** RLMs can theoretically process an infinite amount of data because the LLM only ever sees code and small data chunks, never the whole context.
- **State Management:** By using a REPL, the model maintains a 'state' across iterations, which is more reliable than the 'KV cache' used in standard long-context transformers.
- **Cost Efficiency:** RLMs reduce inference costs by up to 90% for large-scale retrieval and reasoning tasks because they minimize token overhead.

### 3. Further Resources
- **Original Paper:** [Recursive Language Models (arXiv:2512.24601)](https://arxiv.org/abs/2512.24601)
- **DSPy Framework:** For advanced programmatic prompt optimization. [DSPy GitHub](https://github.com/stanfordnlp/dspy)
- **OpenRouter Models:** To explore different model rankings for agentic performance. [OpenRouter Rankings](https://openrouter.ai/rankings)